# Local File Processing to S3

> **Note:** Notebooks in this directory use CLI commands (`!command`) rather than Python library imports. This keeps workflows simple, reproducible, and avoids environment issues with GDAL/rasterio imports.

Process local GeoTIFF files, convert to Cloud Optimized GeoTIFFs (COGs), and upload to S3.

## Available CLI Tools
- `aws s3 ls / cp` - List and transfer S3 files
- `rio cogeo create` - Convert GeoTIFF to Cloud Optimized GeoTIFF
- `rio cogeo validate` - Validate COG structure
- `rio warp` - Reproject rasters
- `gdalinfo -json` - Inspect GeoTIFF metadata

## Workflow
1. Configure local directory and S3 destination
2. List local .tif files
3. Define filename transformations
4. Preview transformations
5. (Optional) Save mapping to CSV
6. Process and upload files
7. Results summary

---

## Step 1: Configuration

Set your local directory path, S3 destination, and processing options:

In [13]:
# ========================================
# INPUTS - Edit these for your event
# ========================================

# Local File Path
LOCAL_DIR = '.'  # Change this to your local directory

# S3 Configuration
BUCKET = 'nasa-disasters'    # S3 bucket (DO NOT CHANGE)
DESTINATION_BASE = 'drcs_activations_new'  # Where to save COGs in S3

# Event Details
EVENT_NAME = '202501_Fire_CA'  # Your event name

# Processing Options
OVERWRITE = False          # Set to True to replace existing files in S3
COMPRESSION = 'zstd'       # COG compression profile (zstd, lzw, deflate)
DST_CRS = 'EPSG:4326'     # Target CRS for reprojection
OVERVIEW_LEVEL = '5'       # Number of overview levels
NODATA = '0'               # Nodata value for COG creation

print(f"Local Directory: {LOCAL_DIR}")
print(f"Event: {EVENT_NAME}")
print(f"Destination: s3://{BUCKET}/{DESTINATION_BASE}/")

Local Directory: .
Event: 202501_Fire_CA
Destination: s3://nasa-disasters/drcs_activations_new/


## Step 2: List Local Files

Scan the local directory for GeoTIFF files:

In [14]:
from pathlib import Path
import os

print("SCANNING LOCAL DIRECTORY")
print("=" * 80)
print(f"\nSearching for .tif files in: {LOCAL_DIR}\n")

local_dir_path = Path(LOCAL_DIR)
if not local_dir_path.exists():
    print(f"ERROR: Directory does not exist: {LOCAL_DIR}")
    print("   Please check your LOCAL_DIR path and try again.")
    local_files = []
else:
    # Find all .tif files recursively (case-insensitive)
    local_files = sorted(
        list(local_dir_path.rglob('*.tif')) + list(local_dir_path.rglob('*.TIF'))
    )

    if local_files:
        print(f"Found {len(local_files)} .tif files\n")
        total_gb = 0
        for i, fp in enumerate(local_files):
            size_gb = fp.stat().st_size / (1024 ** 3)
            total_gb += size_gb
            print(f"  {i+1:3}. {fp.name:<60} ({size_gb:.3f} GB)")
        print(f"\nTotal size: {total_gb:.2f} GB")
    else:
        print("No .tif files found in the specified directory.")
        print("   Check your LOCAL_DIR path.")

print("\n" + "=" * 80)

SCANNING LOCAL DIRECTORY

Searching for .tif files in: .

Found 11 .tif files

    1. earlylook_2025-01-11T19:46:16Z_to_2025-01-11T19:52:10Z.tif   (0.057 GB)
    2. earlylook_2025-01-11T19:59:28Z_to_2025-01-11T20:27:32Z.tif   (0.142 GB)
    3. earlylook_2025-01-11T20:42:30Z_to_2025-01-11T20:48:53Z.tif   (0.059 GB)
    4. earlylook_2025-01-11T20:56:19Z_to_2025-01-11T21:19:29Z.tif   (0.146 GB)
    5. earlylook_2025-01-16T18:53:55Z_to_2025-01-16T18:53:55Z.tif   (0.032 GB)
    6. earlylook_2025-01-16T19:00:58Z_to_2025-01-16T19:00:58Z.tif   (0.023 GB)
    7. earlylook_2025-01-16T19:04:42Z_to_2025-01-16T19:04:42Z.tif   (0.020 GB)
    8. earlylook_2025-01-16T19:09:28Z_to_2025-01-16T19:09:28Z.tif   (0.030 GB)
    9. earlylook_2025-01-16T19:15:22Z_to_2025-01-16T19:38:40Z.tif   (0.143 GB)
   10. earlylook_2025-01-16T19:47:10Z_to_2025-01-16T19:47:10Z.tif   (0.020 GB)
   11. earlylook_2025-01-16T19:52:46Z_to_2025-01-16T20:31:28Z.tif   (0.155 GB)

Total size: 0.83 GB



## Step 3: Define Filename Transformations

Configure how files should be renamed and where each category goes in S3:

In [15]:
# Ordered most-specific -> least-specific. First match wins.
DATETIME_PATTERNS = [
    (r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z?', 'hour'),  # 2025-01-11T19:46:16Z
    (r'\d{8}T\d{6}Z?',                          'hour'),  # 20250111T194616Z
    (r'\d{4}-\d{2}-\d{2}T\d{2}',                'hour'),  # 2025-01-11T19
    (r'\d{8}T\d{2}',                            'hour'),  # 20250111T19
    (r'\d{4}-\d{2}-\d{2}',                      'day'),   # 2025-01-11
    (r'\d{8}',                                  'day'),   # 20250111
]


# Categorization: regex pattern -> S3 output subdirectory
CATEGORIES = {
    r'trueColor|truecolor|true_color|RGB': 'Sentinel-2/trueColor',
    r'colorInfrared|colorIR|color_infrared|CIR': 'Sentinel-2/colorIR',
    r'naturalColor|naturalcolor|natural_color': 'Sentinel-2/naturalColor',
    r'shortwaveIR|SWIR|shortwave': 'Sentinel-2/shortwaveIR',
    r'earlylook': 'AVIRIS',
}


def categorize_file(filename):
    """Match filename to a category. Returns the S3 subdirectory."""
    for pattern, directory in CATEGORIES.items():
        if re.search(pattern, filename, re.IGNORECASE):
            return directory
    return 'uncategorized'


def extract_datetime_from_filename(filename):
    """Return (matched_string, granularity) — granularity is 'hour' or 'day'."""
    for pattern, granularity in DATETIME_PATTERNS:
        m = re.search(pattern, filename)
        if m:
            return m.group(0), granularity
    return None, None


def no_change(original_path, event_name):
    """Pass-through: prepend event name, preserve stem and extension."""
    filename = os.path.basename(original_path)
    stem, ext = os.path.splitext(filename)
    return f"{event_name}_{stem}{ext}"


def create_output_filename(original_path, event_name):
    """Create standardized output filename."""
    filename = os.path.basename(original_path)

    # AVIRIS/earlylook carry datetime ranges — pass through untouched
    if categorize_file(filename).startswith('AVIRIS'):
        return no_change(original_path, event_name)

    stem = os.path.splitext(filename)[0]
    matched, granularity = extract_datetime_from_filename(stem)
    if matched:
        stem_clean = re.sub(r'_?' + re.escape(matched), '', stem, count=1)
        stem_clean = stem_clean.strip('_')
        return f"{event_name}_{stem_clean}_{matched}_{granularity}.tif"
    return f"{event_name}_{stem}_day.tif"


if __name__ == "__main__":
    print("Transformation functions defined")
    print(f"Categories configured: {len(CATEGORIES)}")
    for pattern, directory in CATEGORIES.items():
        print(f"   {directory:<30}  pattern: {pattern}")

Transformation functions defined
Categories configured: 5
   Sentinel-2/trueColor            pattern: trueColor|truecolor|true_color|RGB
   Sentinel-2/colorIR              pattern: colorInfrared|colorIR|color_infrared|CIR
   Sentinel-2/naturalColor         pattern: naturalColor|naturalcolor|natural_color
   Sentinel-2/shortwaveIR          pattern: shortwaveIR|SWIR|shortwave
   AVIRIS                          pattern: earlylook


## Step 4: Preview Transformations

Review old -> new filename mappings before processing:

In [16]:
# Build processing plan with old -> new mappings
processing_plan = []

if local_files:
    print("PREVIEW TRANSFORMATIONS")
    print("=" * 80)

    uncategorized_count = 0
    for fp in local_files:
        filename = fp.name
        new_name = create_output_filename(str(fp), EVENT_NAME)
        output_dir = categorize_file(filename)
        dest_key = f"{DESTINATION_BASE}/{output_dir}/{new_name}"

        processing_plan.append({
            'local_path': str(fp),
            'filename': filename,
            'new_name': new_name,
            'category': output_dir,
            'dest_key': dest_key,
        })

        tag = f"[{output_dir}]" if output_dir != 'uncategorized' else '[UNCATEGORIZED]'
        print(f"\n  {tag}")
        print(f"  Old: {filename}")
        print(f"  New: s3://{BUCKET}/{dest_key}")

        if output_dir == 'uncategorized':
            uncategorized_count += 1

    print(f"\n{'=' * 80}")
    print(f"Total: {len(processing_plan)} files")
    print(f"  Categorized:   {len(processing_plan) - uncategorized_count}")
    print(f"  Uncategorized: {uncategorized_count}")

    if uncategorized_count:
        print("\nAdd patterns to CATEGORIES dict in Step 3 to categorize unmatched files.")
else:
    print("No files to preview. Check Step 2.")

PREVIEW TRANSFORMATIONS

  [AVIRIS]
  Old: earlylook_2025-01-11T19:46:16Z_to_2025-01-11T19:52:10Z.tif
  New: s3://nasa-disasters/drcs_activations_new/AVIRIS/202501_Fire_CA_earlylook_2025-01-11T19:46:16Z_to_2025-01-11T19:52:10Z.tif

  [AVIRIS]
  Old: earlylook_2025-01-11T19:59:28Z_to_2025-01-11T20:27:32Z.tif
  New: s3://nasa-disasters/drcs_activations_new/AVIRIS/202501_Fire_CA_earlylook_2025-01-11T19:59:28Z_to_2025-01-11T20:27:32Z.tif

  [AVIRIS]
  Old: earlylook_2025-01-11T20:42:30Z_to_2025-01-11T20:48:53Z.tif
  New: s3://nasa-disasters/drcs_activations_new/AVIRIS/202501_Fire_CA_earlylook_2025-01-11T20:42:30Z_to_2025-01-11T20:48:53Z.tif

  [AVIRIS]
  Old: earlylook_2025-01-11T20:56:19Z_to_2025-01-11T21:19:29Z.tif
  New: s3://nasa-disasters/drcs_activations_new/AVIRIS/202501_Fire_CA_earlylook_2025-01-11T20:56:19Z_to_2025-01-11T21:19:29Z.tif

  [AVIRIS]
  Old: earlylook_2025-01-16T18:53:55Z_to_2025-01-16T18:53:55Z.tif
  New: s3://nasa-disasters/drcs_activations_new/AVIRIS/202501_Fire_CA_

## Step 5: Optional CSV Export

Save the filename mapping to a CSV file for your records:

In [17]:
import pandas as pd
from datetime import datetime

SAVE_CSV = True  # Set to False to skip CSV export

if SAVE_CSV and processing_plan:
    df = pd.DataFrame(processing_plan)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    csv_path = f'/tmp/{EVENT_NAME}_file_mapping_{timestamp}.csv'
    df.to_csv(csv_path, index=False)
    print(f"Mapping saved to: {csv_path}")
    print(f"Records: {len(df)}")
elif not processing_plan:
    print("No files to export. Check previous steps.")
else:
    print("CSV export disabled (SAVE_CSV = False)")

Mapping saved to: /tmp/202501_Fire_CA_file_mapping_20260515_134836.csv
Records: 11


## Step 6: Process and Upload

For each file: check CRS, reproject if needed, convert to COG, validate, and upload to S3:

In [ ]:
import subprocess, time, json

results = []

for i, item in enumerate(processing_plan):
    local_path = item['local_path']
    dest_key = item['dest_key']
    filename = item['filename']
    new_name = item['new_name']

    print(f"[{i+1}/{len(processing_plan)}] Processing: {filename}")
    start = time.time()

    # --- Check if destination already exists ---
    if not OVERWRITE:
        check = subprocess.run(
            ['aws', 's3', 'ls', f's3://{BUCKET}/{dest_key}'],
            capture_output=True, text=True
        )
        if check.returncode == 0 and check.stdout.strip():
            print(f"  SKIPPED (already exists in S3)\n")
            results.append({'file': filename, 'status': 'skipped', 'time': 0})
            continue

    local_warped = f'/tmp/warped_{new_name}'
    local_cog = f'/tmp/{new_name}'

    try:
        # --- Check CRS with gdalinfo -json ---
        info = subprocess.run(
            ['gdalinfo', '-json', local_path],
            capture_output=True, text=True, check=True
        )
        metadata = json.loads(info.stdout)

        # Determine if reprojection is needed
        crs_wkt = json.dumps(metadata.get('coordinateSystem', {}).get('wkt', ''))
        needs_reproject = DST_CRS.split(':')[-1] not in crs_wkt

        # --- Reproject if needed ---
        if needs_reproject:
            print(f"  Reprojecting to {DST_CRS}...")
            subprocess.run([
                'rio', 'warp', local_path, local_warped,
                '--dst-crs', DST_CRS,
                '--resampling', 'cubic'
            ], capture_output=True, text=True, check=True)
            cog_input = local_warped
        else:
            cog_input = local_path

        # --- Convert to COG ---
        print(f"  Converting to COG ({COMPRESSION})...")
        subprocess.run([
            'rio', 'cogeo', 'create', cog_input, local_cog,
            '--cog-profile', COMPRESSION,
            '--overview-level', OVERVIEW_LEVEL,
            '--nodata', NODATA,
        ], capture_output=True, text=True, check=True)

        # --- Validate COG ---
        val = subprocess.run(
            ['rio', 'cogeo', 'validate', local_cog],
            capture_output=True, text=True
        )
        is_valid = val.returncode == 0
        if not is_valid:
            print(f"  WARNING: COG validation issue: {val.stdout.strip()}")

        # --- Upload to S3 ---
        print(f"  Uploading to s3://{BUCKET}/{dest_key}...")
        subprocess.run(
            ['aws', 's3', 'cp', local_cog, f's3://{BUCKET}/{dest_key}'],
            capture_output=True, text=True, check=True
        )

        elapsed = time.time() - start
        status = 'success' if is_valid else 'success (COG validation warning)'
        print(f"  {status} ({elapsed:.1f}s)\n")
        results.append({'file': filename, 'status': status, 'time': elapsed})

    except subprocess.CalledProcessError as e:
        elapsed = time.time() - start
        err_msg = e.stderr[:300] if e.stderr else str(e)
        print(f"  FAILED: {err_msg}\n")
        results.append({'file': filename, 'status': 'failed', 'time': elapsed, 'error': err_msg})

    finally:
        # Cleanup temp files
        for f in [local_warped, local_cog]:
            if os.path.exists(f):
                os.remove(f)

print("=" * 80)
print("Processing complete.")

[1/11] Processing: earlylook_2025-01-11T19:46:16Z_to_2025-01-11T19:52:10Z.tif
  Converting to COG (zstd)...
  Uploading to s3://nasa-disasters/drcs_activations_new/AVIRIS/202501_Fire_CA_earlylook_2025-01-11T19:46:16Z_to_2025-01-11T19:52:10Z.tif...


## Step 7: Results Summary

Display processing statistics:

In [ ]:
if results:
    total = len(results)
    success = sum(1 for r in results if 'success' in r['status'])
    failed = sum(1 for r in results if r['status'] == 'failed')
    skipped = sum(1 for r in results if r['status'] == 'skipped')

    print("RESULTS SUMMARY")
    print("=" * 80)
    print(f"\nTotal files:  {total}")
    print(f"  Success:    {success}")
    print(f"  Failed:     {failed}")
    print(f"  Skipped:    {skipped}")

    if total > 0:
        print(f"\nSuccess rate: {(success / total * 100):.1f}%")

    if success > 0:
        times = [r['time'] for r in results if 'success' in r['status']]
        print(f"\nTiming:")
        print(f"  Average: {sum(times) / len(times):.1f}s per file")
        print(f"  Slowest: {max(times):.1f}s")
        print(f"  Total:   {sum(times):.1f}s ({sum(times) / 60:.1f} minutes)")

    if failed > 0:
        print(f"\nFailed files:")
        for r in results:
            if r['status'] == 'failed':
                print(f"  - {r['file']}: {r.get('error', 'Unknown')}")

    print("\n" + "=" * 80)
else:
    print("No results to display. Run Step 6 first.")

## Troubleshooting

- **"Directory does not exist"** - Check `LOCAL_DIR` path is correct; use absolute paths
- **"No .tif files found"** - Verify files have `.tif` extension; check subdirectories
- **Files being skipped** - Set `OVERWRITE = True` in Step 1
- **Processing errors** - Check `aws configure` has valid credentials; verify `/tmp` disk space
- **Wrong CRS** - Inspect with `!gdalinfo /tmp/yourfile.tif` and adjust `DST_CRS`
- **COG validation warnings** - Usually harmless; check with `!rio cogeo validate /tmp/yourfile.tif`